# Module 2 — Deploy the agent to AgentCore Runtime (with observability)

In Module 1 you ran the text-to-SQL agent locally. Now you'll deploy the **same agent** to
**Amazon Bedrock AgentCore Runtime** so it runs as a managed, HTTP-invocable service — and you'll
see its execution **traced in CloudWatch**, with no extra agent code.

Unlike a "here's a pre-built project, just deploy it" walkthrough, this notebook builds the whole
thing **end-to-end with the AgentCore CLI** so you see where every piece comes from:

`agentcore create` (scaffold) → configure → `agentcore deploy` → `agentcore invoke` → `agentcore traces`


## How the deploy reuses Module 1 — no rewrite

The deployed agent's brain is the **same `build_agent_options()`** from Module 1's `agent.py`.
The only new file is a thin entrypoint, `analytics_agent/agent_agentcore.py`:

- it's decorated with `@app.entrypoint` (the AgentCore HTTP contract),
- it calls `build_agent_options()` — the single source of truth — with **zero** agent logic of its own,
- it adds deploy-only plumbing: uploading any files the agent produces to S3 and returning presigned URLs.

A drift-guard test asserts this bundle's `agent.py` is byte-identical to Module 1's.

## Setup

`setup.sh` installs the AgentCore CLI (Node), runs `npm ci` for the CDK project, syncs the Python
env, and registers the kernel. Run it once, then select the **agentic-analytics-module-2-deploy**
kernel.

In [ ]:
!bash setup.sh

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

import boto3
acct = boto3.client("sts").get_caller_identity()["Account"]
region = os.getenv("AWS_REGION", "us-west-2")
print("Account:", acct, "| Region:", region)

## Step 1 — Scaffold the AgentCore project (`agentcore create`)

`agentcore create` lays down the project skeleton: `agentcore/agentcore.json` (the resource spec)
and `agentcore/cdk/` (the CDK app that provisions everything). We use `--no-agent` because we'll
attach our **existing** `analytics_agent/` bundle in the next step rather than generating a template.

> **Note:** this repo already ships a known-good `agentcore/` so the module works out of the box.
> The cell below shows the command you'd run from scratch (commented) — running it again is harmless
> but not required.

In [ ]:
# From scratch you'd run (in a fresh project):
# !agentcore create --project-name aaanalytics --no-agent --skip-git --skip-install
#
# Then attach the existing bundle as a "bring-your-own" Container agent:
# !agentcore add agent --name analytics --type byo --language Python \
#     --framework Strands --model-provider Bedrock --build Container --protocol HTTP \
#     --code-location analytics_agent --entrypoint agent_agentcore.py
#
# (Both are already done — agentcore/agentcore.json is committed. Inspect it below.)
import json
print(json.dumps(json.load(open("agentcore/agentcore.json"))["runtimes"][0], indent=2))

## Step 2 — Configure: observability + Bedrock + Athena

Two things make this runtime work:

1. **`instrumentation.enableOtel: true`** — this single flag turns on tracing. The CLI's container
   template runs the agent under `opentelemetry-instrument`, and `agentcore deploy` enables
   **CloudWatch Transaction Search** for you. No hand-written spans, no manual console toggle.
2. **`envVars`** — `CLAUDE_CODE_USE_BEDROCK=1`, the Bedrock model id, and `ATHENA_DATABASE`. The
   Athena results bucket is **derived from your account id at runtime**, so nothing account-specific
   is baked into committed config.

The agent also needs Athena/Glue/S3 permissions at runtime. The CDK auto-creates the runtime role
with only Bedrock + Logs; we add the data-plane permissions in `agentcore/cdk/lib/cdk-stack.ts`, so
**every deploy gets them automatically**.

In [ ]:
# Set your deployment target (account + region). aws-targets.json is gitignored.
import json
targets = [{"name": "default", "description": "my target",
            "account": acct, "region": region}]
with open("agentcore/aws-targets.json", "w") as f:
    json.dump(targets, f, indent=2)
print("wrote agentcore/aws-targets.json →", acct, region)

## Step 3 — (Optional) Run it locally first: `agentcore dev`

`agentcore dev` runs the container's HTTP contract locally so you can smoke-test before deploying.
It's a long-running server, so run it in a **terminal** (not a notebook cell):

```bash
agentcore dev
# then in another shell:
curl -XPOST localhost:8080/invocations -d '{"prompt":"How many students are enrolled?"}'
```

## Step 4 — Deploy (`agentcore deploy`)

This builds the container in the cloud (CodeBuild, ARM64 — no local Docker needed), provisions the
runtime + IAM role via CDK, and enables Transaction Search. Takes a few minutes.

> Your account/region must be CDK-bootstrapped once: `npx cdk bootstrap aws://<account>/<region>`.

In [ ]:
!agentcore deploy -y

## Step 5 — Invoke the deployed agent (`agentcore invoke`)

Send a real question. Use a session id of **≥33 characters** (a runtime requirement). The agent
loads a skill, reads table metadata, writes SQL, runs it on Athena, and answers.

In [ ]:
!agentcore invoke '{"prompt": "How many distinct students are enrolled? Give me the number."}' \
    --session-id agentic-analytics-m2-demo-session-0001

In [ ]:
!agentcore status

## Step 6 — See the trace (`agentcore traces`)

Because `enableOtel` was on, the invocation emitted a trace to **CloudWatch GenAI Observability** —
no extra code. List recent traces (and get the console deep-link); spans take ~2-3 minutes to index.

In [ ]:
!agentcore traces list --runtime analytics --since 1h

## Step 7 — Clean up

Remove the agent and tear down the stack when you're done (avoids ongoing charges).

In [ ]:
# Remove the agent from the project, then deploy to apply the teardown:
# !agentcore remove agent --name analytics -y && agentcore deploy -y
#
# Or destroy the whole CloudFormation stack directly:
# !aws cloudformation delete-stack --stack-name AgentCore-aaanalytics-default --region {region}
print("Uncomment a teardown line above when you're finished.")

## Recap

- The **same** `build_agent_options()` runs locally (Module 1) and deployed — the entrypoint is a
  thin `@app.entrypoint` wrapper, no agent logic duplicated.
- **Observability = one flag.** `enableOtel: true` + `agentcore deploy` gives you CloudWatch traces
  with zero hand-written spans, zero manual console steps.
- The runtime's Athena/Glue/S3 permissions are added in the CDK stack, so deploys are reproducible.

**Next:** Module 3 adds a follow-up-questions (clarification) workflow to the same agent.